# 音のプログラミング 第6回: MIDIとシーケンサー — 既存曲の打ち込みと編曲

このレッスンでは、MIDIの基礎とシーケンサーを使った音楽制作を学びます。

**ゴール:** 既存の楽曲をMIDIデータとして打ち込み、メロディ・伴奏・ベース・ドラムを組み合わせたアレンジを作れるようになる。

**所要時間:** 90分

## 🛠️ 環境セットアップ

In [ ]:
# 🛠️ 環境セットアップ
import sys
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

try:
    import google.colab
    IN_COLAB = True
    print("🔧 Google Colab環境で実行中...")
except ImportError:
    IN_COLAB = False
    print("🏠 ローカル環境で実行中")

if IN_COLAB:
    print("🔧 Google Colab環境を設定中...")
    !pip install japanize-matplotlib
    !git clone https://github.com/ggszk/simple-sound-programming.git
    sys.path.append('/content/simple-sound-programming')
    import japanize_matplotlib
else:
    print("🔧 ローカル環境を設定中...")
    sys.path.append('..')
    import platform
    if platform.system() == 'Darwin':
        plt.rcParams['font.family'] = 'Hiragino Sans'
    else:
        plt.rcParams['font.family'] = 'Meiryo'

print("✅ セットアップ完了")

## 🛠️ 今回使用するライブラリ

In [ ]:
from audio_lib import (
    sine_wave, adsr, AudioSignal,
    note_to_frequency, note_name_to_number, number_to_note_name,
    Sequencer, Note, Track, create_simple_melody, create_chord,
    BasicPiano, BasicOrgan, BasicGuitar, BasicDrum,
)
from audio_lib.notebook import play_sound

## 🎯 実習1: MIDI の基礎知識

**MIDI** (Musical Instrument Digital Interface) は、音楽の演奏データを表現する規格です。
音声そのものではなく、「何の音を」「どの強さで」「いつ」「どれだけの長さ」鳴らすかを記録します。

### MIDI ノート番号

| ノート番号 | 音名 | 説明 |
|-----------|------|------|
| 60 | C4 | 中央のド |
| 62 | D4 | レ |
| 64 | E4 | ミ |
| 65 | F4 | ファ |
| 67 | G4 | ソ |
| 69 | A4 | ラ (= 440Hz) |
| 71 | B4 | シ |
| 72 | C5 | 高いド |

### MIDI の主な要素

- **ノート番号**: 0-127 の値で音の高さを表す
- **ベロシティ**: 0-127 の値で音の強さを表す
- **開始時間**: 音が鳴り始めるタイミング（秒）
- **デュレーション**: 音の長さ（秒）

このライブラリでは、ノート番号の代わりに `"C4"` のような音名文字列も使えます。

In [ ]:
# MIDIノート番号と音名の対応を確認
note_names = ["C4", "D4", "E4", "F4", "G4", "A4", "B4", "C5"]

print("音名  | MIDIノート番号 | 周波数 (Hz)")
print("-" * 40)
for name in note_names:
    num = note_name_to_number(name)
    freq = note_to_frequency(num)
    print(f" {name}  |      {num:3d}       | {freq:.1f}")

In [ ]:
# Note クラスの使い方
note1 = Note("C4", velocity=100, start_time=0.0, duration=0.5)
note2 = Note(60, velocity=100, start_time=0.0, duration=0.5)  # 同じ音

print(f"音名で作成:        {note1}")
print(f"ノート番号で作成:  {note2}")
print(f"周波数:            {note1.get_frequency():.1f} Hz")

## 🎯 実習2: シーケンサーで「きらきら星」を打ち込む

### きらきら星のメロディ (ハ長調)

```
ド ド ソ ソ ラ ラ ソ ー  ファ ファ ミ ミ レ レ ド ー
C  C  G  G  A  A  G  -   F    F   E  E  D  D  C  -
```

### シーケンサーの基本ワークフロー

1. `Sequencer` を作成
2. `Track` を作成し、音符 (`Note`) を追加
3. 楽器を設定
4. `render()` で音声に変換

In [ ]:
# 方法1: add_notes で細かく指定
seq = Sequencer()
melody = Track(name="melody")

melody_notes = [
    # 1フレーズ目: ド ド ソ ソ ラ ラ ソ(伸ばし)
    ("C4", 100, 0.0, 0.5),
    ("C4", 100, 0.5, 0.5),
    ("G4", 100, 1.0, 0.5),
    ("G4", 100, 1.5, 0.5),
    ("A4", 100, 2.0, 0.5),
    ("A4", 100, 2.5, 0.5),
    ("G4", 100, 3.0, 1.0),   # 2分音符
    # 2フレーズ目: ファ ファ ミ ミ レ レ ド(伸ばし)
    ("F4", 100, 4.0, 0.5),
    ("F4", 100, 4.5, 0.5),
    ("E4", 100, 5.0, 0.5),
    ("E4", 100, 5.5, 0.5),
    ("D4", 100, 6.0, 0.5),
    ("D4", 100, 6.5, 0.5),
    ("C4", 100, 7.0, 1.0),
]
melody.add_notes(melody_notes)
melody.set_instrument(BasicPiano())
seq.add_track(melody)

audio = seq.render()
print(f"演奏時間: {len(audio.data) / 44100:.1f} 秒")
display(play_sound(audio, "きらきら星（add_notes版）"))

### create_simple_melody ヘルパー関数

同じ長さの音符が続く場合、`create_simple_melody` を使うとより簡潔に書けます。
`None` を入れると休符になります。

In [ ]:
# 方法2: create_simple_melody で簡潔に書く
seq2 = Sequencer()
melody2 = Track(name="melody")

twinkle_notes = [
    "C4", "C4", "G4", "G4", "A4", "A4", "G4", None,
    "F4", "F4", "E4", "E4", "D4", "D4", "C4", None,
]

create_simple_melody(melody2, twinkle_notes, note_duration=0.5)
melody2.set_instrument(BasicPiano())
seq2.add_track(melody2)

audio2 = seq2.render()
display(play_sound(audio2, "きらきら星（create_simple_melody版）"))

## 🎯 実習3: マルチトラック編曲 — ベースとドラムを追加

メロディだけでは寂しいので、**ベース**と**ドラム**を追加してみましょう。

### ベースライン
コード進行のルート音を低い音域で鳴らします（C - G - F - C）。

### ドラムパターン
ドラムはノート番号が打楽器の種類を表します：
- **36**: キックドラム（バスドラム）
- **38**: スネアドラム
- **42**: ハイハット

In [ ]:
seq3 = Sequencer()

# --- メロディトラック（きらきら星 フル） ---
melody3 = Track(name="melody")
twinkle_full = [
    "C4", "C4", "G4", "G4", "A4", "A4", "G4", None,
    "F4", "F4", "E4", "E4", "D4", "D4", "C4", None,
    "G4", "G4", "F4", "F4", "E4", "E4", "D4", None,
    "G4", "G4", "F4", "F4", "E4", "E4", "D4", None,
    "C4", "C4", "G4", "G4", "A4", "A4", "G4", None,
    "F4", "F4", "E4", "E4", "D4", "D4", "C4", None,
]
create_simple_melody(melody3, twinkle_full, note_duration=0.5)
melody3.set_instrument(BasicPiano())
melody3.volume = 0.7
seq3.add_track(melody3)

# --- ベーストラック ---
bass = Track(name="bass")
bass_pattern = [
    ("C2", 80, 0.0, 2.0),  ("G2", 80, 2.0, 2.0),
    ("F2", 80, 4.0, 2.0),  ("C2", 80, 6.0, 2.0),
    ("C2", 80, 8.0, 2.0),  ("F2", 80, 10.0, 2.0),
    ("C2", 80, 12.0, 2.0), ("G2", 80, 14.0, 2.0),
    ("C2", 80, 16.0, 2.0), ("G2", 80, 18.0, 2.0),
    ("F2", 80, 20.0, 2.0), ("C2", 80, 22.0, 2.0),
]
bass.add_notes(bass_pattern)
bass.set_instrument(BasicGuitar())
bass.volume = 0.5
seq3.add_track(bass)

# --- ドラムトラック ---
drums = Track(name="drums")
total_bars = 12
beat_duration = 0.5

for bar in range(total_bars):
    bar_start = bar * 4 * beat_duration
    for beat in range(4):
        t = bar_start + beat * beat_duration
        drums.add_note(42, 70, t, 0.25)       # ハイハット: 毎拍
        if beat == 0 or beat == 2:
            drums.add_note(36, 90, t, 0.3)     # キック: 1拍目と3拍目
        if beat == 1 or beat == 3:
            drums.add_note(38, 85, t, 0.25)    # スネア: 2拍目と4拍目

drums.set_instrument(BasicDrum())
drums.volume = 0.4
seq3.add_track(drums)

audio3 = seq3.render()
print(f"トラック数: {len(seq3.tracks)}")
display(play_sound(audio3, "きらきら星（メロディ + ベース + ドラム）"))

## 🎯 実習4: リズムパターン — 伴奏の基本

### 代表的なリズムパターン

- **8ビート**: ロック・ポップスの基本。ハイハットが8分音符で刻む
- **16ビート**: ファンク・R&B。より細かいハイハット
- **ワルツ (3拍子)**: クラシック・民謡。1拍目にキック、2-3拍目にハイハット

In [ ]:
def create_drum_pattern(track, pattern_type="8beat", bars=4, beat_duration=0.5):
    """ドラムパターンを生成してトラックに追加する関数"""
    if pattern_type == "8beat":
        beats_per_bar = 4
        for bar in range(bars):
            bar_start = bar * beats_per_bar * beat_duration
            for beat in range(beats_per_bar):
                t = bar_start + beat * beat_duration
                track.add_note(42, 70, t, 0.15)
                track.add_note(42, 60, t + beat_duration / 2, 0.15)
                if beat == 0 or beat == 2:
                    track.add_note(36, 90, t, 0.3)
                if beat == 1 or beat == 3:
                    track.add_note(38, 85, t, 0.2)

    elif pattern_type == "16beat":
        beats_per_bar = 4
        for bar in range(bars):
            bar_start = bar * beats_per_bar * beat_duration
            for beat in range(beats_per_bar):
                t = bar_start + beat * beat_duration
                for sub in range(4):
                    vel = 70 if sub == 0 else 50
                    track.add_note(42, vel, t + sub * beat_duration / 4, 0.1)
                if beat == 0 or beat == 2:
                    track.add_note(36, 90, t, 0.3)
                if beat == 1 or beat == 3:
                    track.add_note(38, 85, t, 0.2)

    elif pattern_type == "waltz":
        beats_per_bar = 3
        for bar in range(bars):
            bar_start = bar * beats_per_bar * beat_duration
            track.add_note(36, 90, bar_start, 0.3)
            track.add_note(42, 70, bar_start + beat_duration, 0.15)
            track.add_note(42, 70, bar_start + 2 * beat_duration, 0.15)

In [ ]:
# 各パターンを聴き比べ
for ptype in ["8beat", "16beat", "waltz"]:
    seq_demo = Sequencer()
    drum_track = Track(name="drums")
    create_drum_pattern(drum_track, pattern_type=ptype, bars=2)
    drum_track.set_instrument(BasicDrum())
    seq_demo.add_track(drum_track)
    audio_demo = seq_demo.render()
    display(play_sound(audio_demo, f"{ptype} パターン"))

## 🎯 実習5: コード進行 — 和声伴奏の基本

コード（和音）はメロディに厚みを与え、曲の雰囲気を決定します。

### よく使われるコード進行

| 進行名 | コード | 構成音 |
|-------|--------|--------|
| I (トニック) | C | ド - ミ - ソ |
| IV (サブドミナント) | F | ファ - ラ - ド |
| V (ドミナント) | G | ソ - シ - レ |
| vi (マイナー) | Am | ラ - ド - ミ |

**I - V - vi - IV** (C - G - Am - F) は数え切れないほどのヒット曲で使われている進行です。

In [ ]:
def create_chord_accompaniment(track, chord_progression, bars_per_chord=1, beat_duration=0.5):
    """コード進行を伴奏としてトラックに追加"""
    bar_duration = 4 * beat_duration
    chord_duration = bars_per_chord * bar_duration

    for i, chord_notes in enumerate(chord_progression):
        start = i * chord_duration
        create_chord(track, chord_notes, start_time=start, duration=chord_duration, velocity=70)

# I - V - vi - IV 進行をオルガンで
seq_chord = Sequencer()
chord_track = Track(name="chords")
progression = [
    ["C3", "E3", "G3"],   # C (I)
    ["G3", "B3", "D4"],   # G (V)
    ["A3", "C4", "E4"],   # Am (vi)
    ["F3", "A3", "C4"],   # F (IV)
]
create_chord_accompaniment(chord_track, progression)
chord_track.set_instrument(BasicOrgan())
chord_track.volume = 0.5
seq_chord.add_track(chord_track)

audio_chord = seq_chord.render()
display(play_sound(audio_chord, "コード進行: C - G - Am - F"))

In [ ]:
# きらきら星にコード伴奏を追加した完全版
seq_full = Sequencer()

# メロディ
melody_full = Track(name="melody")
twinkle_complete = [
    "C4", "C4", "G4", "G4", "A4", "A4", "G4", None,
    "F4", "F4", "E4", "E4", "D4", "D4", "C4", None,
    "G4", "G4", "F4", "F4", "E4", "E4", "D4", None,
    "G4", "G4", "F4", "F4", "E4", "E4", "D4", None,
    "C4", "C4", "G4", "G4", "A4", "A4", "G4", None,
    "F4", "F4", "E4", "E4", "D4", "D4", "C4", None,
]
create_simple_melody(melody_full, twinkle_complete, note_duration=0.5)
melody_full.set_instrument(BasicPiano())
melody_full.volume = 0.7
seq_full.add_track(melody_full)

# コード伴奏
chords_full = Track(name="chords")
twinkle_chords = [
    ["C3", "E3", "G3"], ["G3", "B3", "D4"],
    ["F3", "A3", "C4"], ["C3", "E3", "G3"],
    ["C3", "E3", "G3"], ["F3", "A3", "C4"],
    ["C3", "E3", "G3"], ["G3", "B3", "D4"],
    ["C3", "E3", "G3"], ["G3", "B3", "D4"],
    ["F3", "A3", "C4"], ["C3", "E3", "G3"],
]
create_chord_accompaniment(chords_full, twinkle_chords)
chords_full.set_instrument(BasicOrgan())
chords_full.volume = 0.3
seq_full.add_track(chords_full)

# ベース
bass_full = Track(name="bass")
bass_notes_full = [
    ("C2", 80, 0.0, 2.0), ("G2", 80, 2.0, 2.0),
    ("F2", 80, 4.0, 2.0), ("C2", 80, 6.0, 2.0),
    ("C2", 80, 8.0, 2.0), ("F2", 80, 10.0, 2.0),
    ("C2", 80, 12.0, 2.0), ("G2", 80, 14.0, 2.0),
    ("C2", 80, 16.0, 2.0), ("G2", 80, 18.0, 2.0),
    ("F2", 80, 20.0, 2.0), ("C2", 80, 22.0, 2.0),
]
bass_full.add_notes(bass_notes_full)
bass_full.set_instrument(BasicGuitar())
bass_full.volume = 0.4
seq_full.add_track(bass_full)

# ドラム
drums_full = Track(name="drums")
create_drum_pattern(drums_full, pattern_type="8beat", bars=12)
drums_full.set_instrument(BasicDrum())
drums_full.volume = 0.35
seq_full.add_track(drums_full)

audio_full = seq_full.render()
print(f"メロディ: ピアノ / コード: オルガン / ベース: ギター / ドラム: 8ビート")
display(play_sound(audio_full, "きらきら星 — フルアレンジ版"))

## 🎯 実習6: 実践 — 「ちょうちょ」のフルアレンジ

### メロディ (ハ長調)
```
ソ ミ ミ ー  ファ レ レ ー  ド レ ミ ファ  ソ ソ ソ ー
ソ ミ ミ ー  ファ レ レ ー  ド ミ ソ ソ   ミ ミ ミ ー
```

In [ ]:
# ステップ1: メロディの打ち込み
seq_chou = Sequencer()

melody_chou = Track(name="melody")
choucho_notes = [
    "G4", "E4", "E4", None, "F4", "D4", "D4", None,
    "C4", "D4", "E4", "F4", "G4", "G4", "G4", None,
    "G4", "E4", "E4", None, "F4", "D4", "D4", None,
    "C4", "E4", "G4", "G4", "E4", "E4", "E4", None,
]
create_simple_melody(melody_chou, choucho_notes, note_duration=0.5)
melody_chou.set_instrument(BasicPiano())
melody_chou.volume = 0.7
seq_chou.add_track(melody_chou)

audio_chou_melody = seq_chou.render()
display(play_sound(audio_chou_melody, "ちょうちょ — メロディのみ"))

In [ ]:
# ステップ2-3: コード伴奏 + ベースを追加
chords_chou = Track(name="chords")
choucho_chords = [
    ["C3", "E3", "G3"], ["G2", "B2", "D3"],
    ["C3", "E3", "G3"], ["C3", "E3", "G3"],
    ["C3", "E3", "G3"], ["G2", "B2", "D3"],
    ["C3", "E3", "G3"], ["C3", "E3", "G3"],
]
create_chord_accompaniment(chords_chou, choucho_chords)
chords_chou.set_instrument(BasicOrgan())
chords_chou.volume = 0.3
seq_chou.add_track(chords_chou)

bass_chou = Track(name="bass")
bass_chou_notes = [
    ("C2", 80, 0.0, 2.0),  ("G2", 80, 2.0, 2.0),
    ("C2", 80, 4.0, 2.0),  ("C2", 80, 6.0, 2.0),
    ("C2", 80, 8.0, 2.0),  ("G2", 80, 10.0, 2.0),
    ("C2", 80, 12.0, 2.0), ("C2", 80, 14.0, 2.0),
]
bass_chou.add_notes(bass_chou_notes)
bass_chou.set_instrument(BasicGuitar())
bass_chou.volume = 0.4
seq_chou.add_track(bass_chou)

audio_chou_2 = seq_chou.render()
display(play_sound(audio_chou_2, "ちょうちょ — メロディ + コード + ベース"))

In [ ]:
# ステップ4-5: ドラムを追加して完成
drums_chou = Track(name="drums")
create_drum_pattern(drums_chou, pattern_type="8beat", bars=8)
drums_chou.set_instrument(BasicDrum())
drums_chou.volume = 0.35
seq_chou.add_track(drums_chou)

audio_chou_final = seq_chou.render()
print(f"メロディ: ピアノ (0.7) / コード: オルガン (0.3) / ベース: ギター (0.4) / ドラム: 8ビート (0.35)")
display(play_sound(audio_chou_final, "ちょうちょ — フルアレンジ版"))

## 🏆 チャレンジ課題

### 課題1: 好きな既存曲の冒頭8小節をメロディとして打ち込む

**ヒント:**
- まず曲を口ずさんで、音の上下関係を把握する
- ハ長調に移調すると打ち込みやすい
- 最初は4分音符だけで近似して、後でリズムを調整する

### 課題2: その曲に伴奏（コード + ベース + ドラム）を付ける

**ヒント:**
- 「曲名 コード進行」で検索するとコードが見つかることが多い
- まずは I (C), IV (F), V (G) の3つだけで試す
- `volume` を調整してバランスを取る

In [ ]:
# 課題1: メロディの打ち込み
seq_kadai = Sequencer()
melody_kadai = Track(name="melody")

# 例: 「かえるの合唱」の冒頭
kadai_notes = [
    "C4", "D4", "E4", "F4", "E4", "D4", "C4", None,
    "E4", "F4", "G4", "A4", "G4", "F4", "E4", None,
]
create_simple_melody(melody_kadai, kadai_notes, note_duration=0.5)
melody_kadai.set_instrument(BasicPiano())
melody_kadai.volume = 0.7
seq_kadai.add_track(melody_kadai)

audio_kadai = seq_kadai.render()
display(play_sound(audio_kadai, "課題1: メロディ"))

In [ ]:
# 課題2: 伴奏を追加
chords_kadai = Track(name="chords")
kadai_chords = [
    ["C3", "E3", "G3"],
    ["G2", "B2", "D3"],
]
create_chord_accompaniment(chords_kadai, kadai_chords)
chords_kadai.set_instrument(BasicOrgan())
chords_kadai.volume = 0.3
seq_kadai.add_track(chords_kadai)

bass_kadai = Track(name="bass")
bass_kadai.add_notes([
    ("C2", 80, 0.0, 2.0),
    ("G2", 80, 2.0, 2.0),
])
bass_kadai.set_instrument(BasicGuitar())
bass_kadai.volume = 0.4
seq_kadai.add_track(bass_kadai)

drums_kadai = Track(name="drums")
create_drum_pattern(drums_kadai, pattern_type="8beat", bars=4)
drums_kadai.set_instrument(BasicDrum())
drums_kadai.volume = 0.35
seq_kadai.add_track(drums_kadai)

audio_kadai_full = seq_kadai.render()
display(play_sound(audio_kadai_full, "課題2: フルアレンジ"))

## 📚 今日のまとめ

### 学んだこと
- MIDIは「何の音を、いつ、どれだけの長さで鳴らすか」を記録する規格
- `Note` でMIDI音符を、`Track` でパートを、`Sequencer` で楽曲全体を管理
- メロディ、コード、ベース、ドラムを組み合わせてアレンジを作る
- 既存曲の打ち込みは音楽理論を実践的に学ぶ最良の方法

### 次回予告
第7回は**最終プロジェクト**です。
好きな曲を1曲選び、全レッスンで学んだ技術を総動員して再現します。